# Example visualizations

Illustrative plots built from the data and pipeline outputs shipped in this repository. These are examples, not a reproduction of the published figures; the numerical values behind the paper's main and supplementary figures are provided as the manuscript's Source Data.

Run All from the repo root. Examples 1-2 run on the shipped curated data; Example 3 previews the two-stage model outputs if you have run `execute_production.ipynb`.

In [ ]:
import os, sys
ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, 'src')) and os.path.dirname(ROOT) != ROOT:
    ROOT = os.path.dirname(ROOT)
SRC, DATA = os.path.join(ROOT, 'src'), os.path.join(ROOT, 'data')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import utils, simulations as sim
plt.rcParams.update(utils.RCPARAMS)
print('repo root :', ROOT)

## Example 1: activity across LD-BT rounds
Relative specific activity (RebA, x wild-type) of the measured variants in each Design-Build-Test-Learn round. Wild-type reference = 1.0x (dashed). Related to Fig 1.

In [ ]:
df = utils.load_data(os.path.join(DATA, 'SrUGT76G1_variant_library.xlsx'))
rounds = utils.ROUND_ORDER  # LD-BT 0..3
rng = np.random.default_rng(0)
fig, ax = plt.subplots(figsize=(6, 4))
for i, r in enumerate(rounds):
    vals = df.loc[df['Round'] == r, 'RSA_RebA_xWT'].dropna().values
    x = i + rng.uniform(-0.18, 0.18, size=len(vals))
    ax.scatter(x, vals, s=18, alpha=0.7,
               color=utils.ROUND_COLORS.get(r, '#888888'),
               edgecolors='black', linewidths=0.4, label='%s (n=%d)' % (r, len(vals)))
ax.axhline(1.0, ls='--', color='#404040', lw=1.0)
ax.set_xticks(range(len(rounds)))
ax.set_xticklabels(rounds, rotation=20, ha='right')
ax.set_ylabel('RebA relative specific activity (x WT)')
ax.legend(loc='upper left', fontsize=6)
plt.tight_layout()
print('plotted %d measured variants across %d rounds' % (
    df['Round'].isin(rounds).sum(), len(rounds)))
plt.show()

## Example 2: library-construction strategy simulation
Expected per-position mutation frequency (epPCR strategies) and amino-acid diversity (SSM) versus colony-pick budget. Related to Fig 6.

In [ ]:
colony_ns = sorted(set(sim.COLONY_NS))
ep_freq, ssm_uniq = [], []
for N in colony_ns:
    mf, _, _, _ = sim.simulate_eppcr_avg(N, n_seeds=sim.N_SEEDS,
                                         base_seed=sim.EPPCR_AVG_BASE_SEED)
    ep_freq.append(float(np.mean(mf)))
    ssm_uniq.append(sim.ssm_coupon_expected(N))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(8, 3.5))
a1.plot(colony_ns, ep_freq, 'o-', color='#5B9BD5')
a1.set_xlabel('colony picks'); a1.set_ylabel('mean mutation freq (%)')
a1.set_title('epPCR')
a2.plot(colony_ns, ssm_uniq, 's-', color='#52b788')
a2.set_xlabel('colony picks'); a2.set_ylabel('expected unique amino acids')
a2.set_title('SSM (coupon-collector)')
plt.tight_layout()
print('colony budgets simulated:', colony_ns)
plt.show()

## Example 3: two-stage model outputs (if available)
Preview of the CV / validation tables written by `execute_production.ipynb`. Related to Fig 3, Fig 4.

In [ ]:
prod = os.path.join(ROOT, 'results', 'SrUGT76G1_production')
if os.path.isdir(prod):
    files = sorted(f for f in os.listdir(prod) if f.endswith(('.csv', '.xlsx')))
    print('found %d output table(s) in %s' % (len(files), prod))
    for f in files[:10]:
        print('  ', f)
    if files:
        first = os.path.join(prod, files[0])
        prev = pd.read_csv(first) if first.endswith('.csv') else pd.read_excel(first)
        display(prev.head())
else:
    print('No production outputs yet. Run execute_production.ipynb to generate them,')
    print('then re-run this cell. (Requires the per-residue training workbook.)')